In [8]:
pip install  datasets peft trl accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 16.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0


In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

In [2]:

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
model_name = "Qwen/Qwen2.5-3B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16
)



In [4]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(model_name)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


model.safetensors.index.json:   0%|          | 0.00/35.6k [00:00<?, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [6]:
# for qlora training

In [5]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [7]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)

In [8]:
model.print_trainable_parameters()

trainable params: 14,966,784 || all params: 3,100,905,472 || trainable%: 0.4827


In [9]:
from transformers import TrainingArguments
from trl import SFTTrainer

In [10]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files="banking_dataset.json"
)

print(dataset["train"][0])

Generating train split: 0 examples [00:00, ? examples/s]

{'messages': [{'role': 'user', 'content': 'How can I open a new bank account?'}, {'role': 'assistant', 'content': "To open a new bank account, visit our website and click on 'Open Account' or visit any of our branches with your identification documents."}]}


In [11]:
def apply_template(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )

    return {"text": text}

In [12]:
dataset = dataset.map(apply_template)

Map:   0%|          | 0/145 [00:00<?, ? examples/s]

In [20]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=5,
    learning_rate=2e-4,
    logging_steps=10,
    save_steps=100,
    fp16=False,
    bf16=True,
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    processing_class=tokenizer,
    args=training_args,
)

In [15]:
print(dataset["train"][0])

{'messages': [{'role': 'user', 'content': 'How can I open a new bank account?'}, {'role': 'assistant', 'content': "To open a new bank account, visit our website and click on 'Open Account' or visit any of our branches with your identification documents."}], 'text': "<|im_start|>system\nYou are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>\n<|im_start|>user\nHow can I open a new bank account?<|im_end|>\n<|im_start|>assistant\nTo open a new bank account, visit our website and click on 'Open Account' or visit any of our branches with your identification documents.<|im_end|>\n"}


In [16]:
# training

In [21]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1263: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss
10,0.588684
20,0.512212
30,0.368039
40,0.331485
50,0.247037
60,0.243759
70,0.174396
80,0.162958
90,0.152936


TrainOutput(global_step=95, training_loss=0.30063920711216174, metrics={'train_runtime': 301.1533, 'train_samples_per_second': 2.407, 'train_steps_per_second': 0.315, 'total_flos': 798557651238912.0, 'train_loss': 0.30063920711216174, 'entropy': 0.1840876034077476, 'num_tokens': 46285.0, 'mean_token_accuracy': 0.9469373015796437, 'epoch': 5.0})

In [22]:
trainer.save_model("./banking-chatbot-lora")
tokenizer.save_pretrained("./banking-chatbot-lora")

('./banking-chatbot-lora/tokenizer_config.json',
 './banking-chatbot-lora/chat_template.jinja',
 './banking-chatbot-lora/tokenizer.json')

In [23]:
# testing

In [25]:
model = PeftModel.from_pretrained(
    model,
    "./banking-chatbot-lora"
)

tokenizer = AutoTokenizer.from_pretrained(
    "./banking-chatbot-lora"
)

/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/peft_model.py:622: UserWarning: Found missing adapter keys while loading the checkpoint: ['base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.k_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_A.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.v_proj.lora_B.default.weight', 'base_model.model.base_model.model.model.layers.0.self_attn.o_proj.

In [41]:
question = "How can  Iopen a joint account with my mother through my mobile?"

In [42]:
# formatting to give to model

In [47]:
messages = [
    {
        "role": "user",
        "content": question
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

In [48]:
inputs = tokenizer(
    text,
    return_tensors="pt"
).to(model.device)

In [49]:
outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.3,
    do_sample=True,
    pad_token_id=tokenizer.pad_token_id,
)

In [50]:
response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.
user
How can  Iopen a joint account with my mother through my mobile?
assistant
Opening a joint account with your mother through your mobile app typically involves the following steps:

1. **Log in to Your Bank’s Mobile App**: First, you need to log in to your bank's official mobile banking application on your smartphone.

2. **Navigate to Account Management or Joint Accounts Section**: Once logged in, look for an option that says "Account Management," "Joint Accounts," or something similar. This section usually contains options to add new joint accounts.

3. **Add a New Joint


In [51]:
generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

answer = tokenizer.decode(
    generated_tokens,
    skip_special_tokens=True
)

print(answer)

Opening a joint account with your mother through your mobile app typically involves the following steps:

1. **Log in to Your Bank’s Mobile App**: First, you need to log in to your bank's official mobile banking application on your smartphone.

2. **Navigate to Account Management or Joint Accounts Section**: Once logged in, look for an option that says "Account Management," "Joint Accounts," or something similar. This section usually contains options to add new joint accounts.

3. **Add a New Joint


In [52]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [53]:
trainer.save_model("/content/drive/MyDrive/banking-chatbot-lora4")
tokenizer.save_pretrained("/content/drive/MyDrive/banking-chatbot-lora4")

('/content/drive/MyDrive/banking-chatbot-lora4/tokenizer_config.json',
 '/content/drive/MyDrive/banking-chatbot-lora4/chat_template.jinja',
 '/content/drive/MyDrive/banking-chatbot-lora4/tokenizer.json')